In [1]:
from pathlib import Path
import geopandas as gpd

gpx_path = Path(r"C:\RaceGuard\data\raw\peachtree_course_plotaroute_2300594.gpx")

print("Path exists:", gpx_path.exists())

layers = gpd.list_layers(gpx_path)
print(layers)

Path exists: True
           name    geometry_type
0     waypoints            Point
1        routes       LineString
2        tracks  MultiLineString
3  route_points            Point
4  track_points            Point


In [2]:
course_tracks = gpd.read_file(
    gpx_path,
    layer="tracks",
)

geometry_column = course_tracks.geometry.name

attribute_data = course_tracks.drop(
    columns=geometry_column
)

populated_attributes = attribute_data.dropna(
    axis="columns",
    how="all",
)

print(f"Number of track rows: {len(course_tracks)}")
print(f"Columns: {course_tracks.columns.tolist()}")
print(f"CRS: {course_tracks.crs}")

print("\nGeometry types:")
print(
    course_tracks.geom_type
    .value_counts(dropna=False)
    .to_string()
)

print(
    "\nContains missing geometry:",
    course_tracks.geometry.isna().any(),
)

print(
    "Contains empty geometry:",
    course_tracks.geometry.is_empty.any(),
)

print(
    "All geometries valid:",
    course_tracks.geometry.is_valid.all(),
)

print("\nPopulated track attributes:")

if populated_attributes.shape[1] == 0:
    print("No populated non-geometry attributes")
else:
    print(
        populated_attributes.to_string(
            index=False
        )
    )

Number of track rows: 1
Columns: ['name', 'cmt', 'desc', 'src', 'link1_href', 'link1_text', 'link1_type', 'link2_href', 'link2_text', 'link2_type', 'number', 'type', 'geometry']
CRS: EPSG:4326

Geometry types:
MultiLineString    1

Contains missing geometry: False
Contains empty geometry: False
All geometries valid: True

Populated track attributes:
                   name
AJC Peachtree Road Race


In [3]:
OFFICIAL_DISTANCE_KM = 10.0
LENGTH_TOLERANCE_PERCENT = 1.0

metric_crs = course_tracks.estimate_utm_crs()

course_tracks_metric = course_tracks.to_crs(
    metric_crs
)

route_length_metres = (
    course_tracks_metric.geometry.length.sum()
)

route_length_km = route_length_metres / 1000
route_length_miles = route_length_km * 0.621371

difference_metres = abs(
    route_length_metres
    - OFFICIAL_DISTANCE_KM * 1000
)

difference_percent = (
    difference_metres
    / (OFFICIAL_DISTANCE_KM * 1000)
    * 100
)

length_check_passed = (
    difference_percent
    <= LENGTH_TOLERANCE_PERCENT
)

track_geometry = course_tracks.geometry.iloc[0]

if track_geometry.geom_type == "MultiLineString":
    track_parts = list(track_geometry.geoms)
else:
    track_parts = [track_geometry]

coordinate_count = sum(
    len(part.coords)
    for part in track_parts
)

print(f"Original CRS: {course_tracks.crs}")
print(f"Metric CRS: {metric_crs}")
print(f"Track parts: {len(track_parts)}")
print(f"Coordinate count: {coordinate_count:,}")
print(f"Calculated length: {route_length_metres:,.2f} m")
print(f"Calculated length: {route_length_km:.4f} km")
print(f"Calculated length: {route_length_miles:.4f} miles")
print(f"Difference from 10K: {difference_metres:.2f} m")
print(f"Percentage difference: {difference_percent:.3f}%")
print(f"Within {LENGTH_TOLERANCE_PERCENT:.1f}% tolerance: {length_check_passed}")

Original CRS: EPSG:4326
Metric CRS: EPSG:32616
Track parts: 1
Coordinate count: 155
Calculated length: 9,990.25 m
Calculated length: 9.9902 km
Calculated length: 6.2077 miles
Difference from 10K: 9.75 m
Percentage difference: 0.098%
Within 1.0% tolerance: True


In [4]:
course_line = track_parts[0]

start_coordinate = course_line.coords[0]
finish_coordinate = course_line.coords[-1]

start_longitude = start_coordinate[0]
start_latitude = start_coordinate[1]

finish_longitude = finish_coordinate[0]
finish_latitude = finish_coordinate[1]

minimum_longitude, minimum_latitude, maximum_longitude, maximum_latitude = (
    course_tracks.total_bounds
)

bounds_are_in_atlanta = (
    -85.0 <= minimum_longitude <= -84.0
    and -85.0 <= maximum_longitude <= -84.0
    and 33.0 <= minimum_latitude <= 34.5
    and 33.0 <= maximum_latitude <= 34.5
)

route_runs_generally_south = (
    start_latitude > finish_latitude
)

print("Start point:")
print(f"  Latitude: {start_latitude:.6f}")
print(f"  Longitude: {start_longitude:.6f}")

print("\nFinish point:")
print(f"  Latitude: {finish_latitude:.6f}")
print(f"  Longitude: {finish_longitude:.6f}")

print("\nCourse bounds:")
print(f"  Minimum longitude: {minimum_longitude:.6f}")
print(f"  Minimum latitude: {minimum_latitude:.6f}")
print(f"  Maximum longitude: {maximum_longitude:.6f}")
print(f"  Maximum latitude: {maximum_latitude:.6f}")

print(f"\nBounds fall within Atlanta region: {bounds_are_in_atlanta}")
print(f"Route runs generally north-to-south: {route_runs_generally_south}")

Start point:
  Latitude: 33.849322
  Longitude: -84.363280

Finish point:
  Latitude: 33.781792
  Longitude: -84.374667

Course bounds:
  Minimum longitude: -84.393950
  Minimum latitude: 33.781708
  Maximum longitude: -84.363232
  Maximum latitude: 33.849440

Bounds fall within Atlanta region: True
Route runs generally north-to-south: True


In [5]:
# import folium


# map_centre = [
#     (minimum_latitude + maximum_latitude) / 2,
#     (minimum_longitude + maximum_longitude) / 2,
# ]

# validation_map = folium.Map(
#     location=map_centre,
#     zoom_start=13,
#     tiles="OpenStreetMap",
# )

# folium.GeoJson(
#     course_tracks.__geo_interface__,
#     name="Candidate Peachtree course",
#     style_function=lambda feature: {
#         "color": "#1565C0",
#         "weight": 6,
#         "opacity": 0.9,
#     },
#     tooltip="Candidate Peachtree course",
# ).add_to(validation_map)

# folium.Marker(
#     location=[
#         start_latitude,
#         start_longitude,
#     ],
#     tooltip="Course start",
#     popup=(
#         f"Start<br>"
#         f"Latitude: {start_latitude:.6f}<br>"
#         f"Longitude: {start_longitude:.6f}"
#     ),
#     icon=folium.Icon(
#         color="green",
#         icon="play",
#     ),
# ).add_to(validation_map)

# folium.Marker(
#     location=[
#         finish_latitude,
#         finish_longitude,
#     ],
#     tooltip="Course finish",
#     popup=(
#         f"Finish<br>"
#         f"Latitude: {finish_latitude:.6f}<br>"
#         f"Longitude: {finish_longitude:.6f}"
#     ),
#     icon=folium.Icon(
#         color="red",
#         icon="flag",
#     ),
# ).add_to(validation_map)

# validation_map.fit_bounds(
#     [
#         [minimum_latitude, minimum_longitude],
#         [maximum_latitude, maximum_longitude],
#     ]
# )

# folium.LayerControl().add_to(
#     validation_map
# )

# validation_map

In [6]:
from pathlib import Path

import geopandas as gpd


# Locate the repository whether Jupyter started from the repository root
# or from the notebooks directory.
current_directory = Path.cwd()

repo_root = (
    current_directory.parent
    if current_directory.name == "notebooks"
    else current_directory
)

output_path = (
    repo_root
    / "data"
    / "processed"
    / "peachtree_course.geojson"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)


# The validated GPX contains one MultiLineString with one component.
# Extract that component without silently discarding additional parts.
track_geometry = course_tracks.geometry.iloc[0]

if track_geometry.geom_type == "MultiLineString":
    track_parts = list(track_geometry.geoms)

    if len(track_parts) != 1:
        raise ValueError(
            f"Expected one connected track part, found {len(track_parts)}."
        )

    course_line = track_parts[0]
else:
    course_line = track_geometry


if course_line.geom_type != "LineString":
    raise ValueError(
        f"Expected a LineString, found {course_line.geom_type}."
    )


# Build RaceGuard's canonical course dataset.
# We deliberately retain EPSG:4326 because GeoJSON and FortyGuard use
# longitude/latitude coordinates.
canonical_course = gpd.GeoDataFrame(
    {
        "name": ["AJC Peachtree Road Race"],
        "source": ["PlotARoute route 2300594"],
        "validation_status": ["validated"],
    },
    geometry=[course_line],
    crs=course_tracks.crs,
)

canonical_course.to_file(
    output_path,
    driver="GeoJSON",
)


# Read the actual saved file back and validate it.
saved_course = gpd.read_file(output_path)

validation_checks = {
    "file exists": output_path.exists(),
    "one feature": len(saved_course) == 1,
    "LineString geometry": (
        saved_course.geometry.iloc[0].geom_type == "LineString"
    ),
    "EPSG:4326 CRS": saved_course.crs.to_epsg() == 4326,
    "valid geometry": saved_course.geometry.is_valid.all(),
    "non-empty geometry": not saved_course.geometry.is_empty.any(),
}

print("Saved course:", output_path)
print()

for check_name, passed in validation_checks.items():
    print(f"{check_name}: {passed}")

print()
print("Saved properties:")
print(saved_course.drop(columns="geometry").to_string(index=False))

if not all(validation_checks.values()):
    raise ValueError("The canonical course failed validation.")

Saved course: c:\RaceGuard\data\processed\peachtree_course.geojson

file exists: True
one feature: True
LineString geometry: True
EPSG:4326 CRS: True
valid geometry: True
non-empty geometry: True

Saved properties:
                   name                   source validation_status
AJC Peachtree Road Race PlotARoute route 2300594         validated
